In [0]:
-- Step 1, 2, 3은 이전과 동일
-- Step 1: 월별 활동 기록 추출
WITH user_monthly_activity AS (
    SELECT
        DISTINCT
        date_ym AS activity_month,
        case when 
          device_name in ('o22n2', 'o24', 'k24') then 'fast_group'
          ELSE 'slow_group'
        end as device_group,
        mac_addr
    FROM kic_data_ods.tlamp.normal_log_webos24
    WHERE 1=1
         AND date_ym <= '2026-04'
        AND DEVICE_NAME in ('k24', 'k8lpn2', 'o22n2', 'o24')
        AND log_create_time >= '2024-04-01'
        AND X_Device_Country = 'KR'
        AND context_name = 'LSM'
        AND normal_log:app_id = 'com.webos.app.lgchannels'
        AND normal_log:visible = true
),
-- Step 2: 첫 활동 월(코호트) 정의
user_first_activity AS (
    SELECT
        device_group,
        mac_addr,
        MIN(activity_month) AS cohort_month
    FROM user_monthly_activity
    GROUP BY device_group, mac_addr
),
-- Step 3: 코호트 월로부터의 경과 월 계산
cohort_retention_data AS (
    SELECT
        fa.cohort_month,
        fa.device_group,
        fa.mac_addr,
        CAST(FLOOR(months_between(
            TO_DATE(ma.activity_month, 'yyyy-MM'),
            TO_DATE(fa.cohort_month, 'yyyy-MM')
        )) AS INT) AS month_number
    FROM user_first_activity fa
    JOIN user_monthly_activity ma
      ON fa.mac_addr = ma.mac_addr AND fa.device_group = ma.device_group
    WHERE
        fa.cohort_month BETWEEN '2024-04' AND '2026-04'
),

-- Step 4: 코호트별, 월차별 잔존 유저 수(절대값)를 계산합니다. (중간 단계)
cohort_absolute_counts AS (
    SELECT
        cohort_month,
        device_group,
        COUNT(DISTINCT CASE WHEN month_number = 0 THEN mac_addr END) AS M0,
        COUNT(DISTINCT CASE WHEN month_number = 1 THEN mac_addr END) AS M1,
        COUNT(DISTINCT CASE WHEN month_number = 2 THEN mac_addr END) AS M2,
        COUNT(DISTINCT CASE WHEN month_number = 3 THEN mac_addr END) AS M3,
        COUNT(DISTINCT CASE WHEN month_number = 4 THEN mac_addr END) AS M4,
        COUNT(DISTINCT CASE WHEN month_number = 5 THEN mac_addr END) AS M5,
        COUNT(DISTINCT CASE WHEN month_number = 6 THEN mac_addr END) AS M6,
        COUNT(DISTINCT CASE WHEN month_number = 7 THEN mac_addr END) AS M7,
        COUNT(DISTINCT CASE WHEN month_number = 8 THEN mac_addr END) AS M8,
        COUNT(DISTINCT CASE WHEN month_number = 9 THEN mac_addr END) AS M9,
        COUNT(DISTINCT CASE WHEN month_number = 10 THEN mac_addr END) AS M10,
        COUNT(DISTINCT CASE WHEN month_number = 11 THEN mac_addr END) AS M11,
        COUNT(DISTINCT CASE WHEN month_number = 12 THEN mac_addr END) AS M12,
        COUNT(DISTINCT CASE WHEN month_number = 13 THEN mac_addr END) AS M13,
        COUNT(DISTINCT CASE WHEN month_number = 14 THEN mac_addr END) AS M14,
        COUNT(DISTINCT CASE WHEN month_number = 15 THEN mac_addr END) AS M15,
        COUNT(DISTINCT CASE WHEN month_number = 16 THEN mac_addr END) AS M16,
        COUNT(DISTINCT CASE WHEN month_number = 17 THEN mac_addr END) AS M17,
        COUNT(DISTINCT CASE WHEN month_number = 18 THEN mac_addr END) AS M18,
        COUNT(DISTINCT CASE WHEN month_number = 19 THEN mac_addr END) AS M19,
        COUNT(DISTINCT CASE WHEN month_number = 20 THEN mac_addr END) AS M20,
        COUNT(DISTINCT CASE WHEN month_number = 21 THEN mac_addr END) AS M21,
        COUNT(DISTINCT CASE WHEN month_number = 22 THEN mac_addr END) AS M22,
        COUNT(DISTINCT CASE WHEN month_number = 23 THEN mac_addr END) AS M23,
        COUNT(DISTINCT CASE WHEN month_number = 24 THEN mac_addr END) AS M24
    FROM cohort_retention_data
    GROUP BY cohort_month, device_group
)

-- Step 5: 절대값 카운트를 바탕으로 잔존율(%)을 계산하여 최종 결과를 출력합니다.
SELECT
    cohort_month,
    device_group,
    M0 AS initial_users,
    M1 AS M1_retained_users,
    M2 AS M2_retained_users,
    M3 AS M3_retained_users,
    M4 AS M4_retained_users,
    M5 AS M5_retained_users,
    M6 AS M6_retained_users,
    M7 AS M7_retained_users,
    M8 AS M8_retained_users,
    M9 AS M9_retained_users,
    M10 AS M10_retained_users,
    M11 AS M11_retained_users,
    M12 AS M12_retained_users,    
    M13 AS M13_retained_users,
    M14 AS M14_retained_users,
    M15 AS M15_retained_users,
    M16 AS M16_retained_users,
    M17 AS M17_retained_users,
    M18 AS M18_retained_users,
    M19 AS M19_retained_users,
    M20 AS M20_retained_users,
    M21 AS M21_retained_users,
    M22 AS M22_retained_users,  
    M23 AS M23_retained_users,  
    M24 AS M24_retained_users,  
    100 AS intial_rate,
         -- M+1 잔존율
    ROUND(M1 / M0 * 100, 2) AS M1_retention_rate,
    -- M+2 잔존율
    ROUND(M2 / M0 * 100, 2) AS M2_retention_rate,
    -- M+3 잔존율
    ROUND(M3 / M0 * 100, 2) AS M3_retention_rate,
    -- M+4 잔존율
    ROUND(M4 / M0 * 100, 2) AS M4_retention_rate,
    -- M+5 잔존율
    ROUND(M5 / M0 * 100, 2) AS M5_retention_rate,
    -- M+6 잔존율
    ROUND(M6 / M0 * 100, 2) AS M6_retention_rate,
    -- M+7 잔존율
    ROUND(M7 / M0 * 100, 2) AS M7_retention_rate,
    -- M+8 잔존율
    ROUND(M8 / M0 * 100, 2) AS M8_retention_rate,
    -- M+9 잔존율
    ROUND(M9 / M0 * 100, 2) AS M9_retention_rate,
    -- M+10 잔존율
    ROUND(M10 / M0 * 100, 2) AS M10_retention_rate,
    -- M+11 잔존율
    ROUND(M11 / M0 * 100, 2) AS M11_retention_rate,
    -- M+12 잔존율
    ROUND(M12 / M0 * 100, 2) AS M12_retention_rate,
    -- M+13 잔존율
    ROUND(M13 / M0 * 100, 2) AS M13_retention_rate,
    -- M+14 잔존율
    ROUND(M14 / M0 * 100, 2) AS M14_retention_rate,
    -- M+15 잔존율
    ROUND(M15 / M0 * 100, 2) AS M15_retention_rate,
    -- M+16 잔존율
    ROUND(M16 / M0 * 100, 2) AS M16_retention_rate,
    -- M+17 잔존율
    ROUND(M17 / M0 * 100, 2) AS M17_retention_rate,
    -- M+18 잔존율
    ROUND(M18 / M0 * 100, 2) AS M18_retention_rate,
    -- M+19 잔존율
    ROUND(M19 / M0 * 100, 2) AS M19_retention_rate,
    -- M+20 잔존율
    ROUND(M20 / M0 * 100, 2) AS M20_retention_rate,
    -- M+21 잔존율
    ROUND(M21 / M0 * 100, 2) AS M21_retention_rate,
    -- M+22 잔존율
    ROUND(M22 / M0 * 100, 2) AS M22_retention_rate,
    -- M+23 잔존율
    ROUND(M23 / M0 * 100, 2) AS M23_retention_rate,
    -- M+24 잔존율
    ROUND(M24 / M0 * 100, 2) AS M24_retention_rate
FROM cohort_absolute_counts
-- 0으로 나누는 오류 방지 (신규 유저가 없는 경우 해당 행 제외)
WHERE M0 > 0
ORDER BY device_group, cohort_month;